
# Multimodal Scientific & Medical Data Ingestion

**Objective:** Read and demonstrate ingestion of **tabular, textual, image, signal, and medical data** in different formats using Python.

### Datasets used
- **Tabular/Medical:** UCI Cleveland Heart Disease Dataset
- **Signal:** MIT-BIH Arrhythmia Database (record `100`) via PhysioNet/WFDB
- **Text:** Synthetic clinical note
- **Image:** Synthetic medical-style grayscale image (no external image download required)

The notebook also performs basic validation, preprocessing, visualization, a baseline classifier, and a discussion of multimodal fusion.


In [ ]:

# 1. Environment Setup
!pip -q install pandas numpy scikit-learn matplotlib wfdb pillow openpyxl

import os
import re
import json
import math
import hashlib
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report, RocCurveDisplay
)

import wfdb

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

print("Environment ready.")



## 2. Generalized Multimodal Loader Design

The following loader functions demonstrate how a real ingestion module can dispatch files according to modality and format. The functions are intentionally reusable for future datasets.


In [ ]:

# Generic loader functions

def load_tabular(path):
    """Load CSV, Excel, or JSON tabular data."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return pd.read_csv(path)
    elif ext in [".xls", ".xlsx"]:
        return pd.read_excel(path)
    elif ext == ".json":
        return pd.read_json(path)
    else:
        raise ValueError(f"Unsupported tabular format: {ext}")


def load_text(path, encoding="utf-8"):
    """Load plain text."""
    with open(path, "r", encoding=encoding) as f:
        return f.read()


def load_image(path):
    """Load PNG/JPEG/TIFF images with Pillow."""
    return Image.open(path)


def inspect_dataframe(df):
    """Basic schema and missing-value inspection."""
    print("Shape:", df.shape)
    display(df.head())
    print("\nData types:")
    display(df.dtypes)
    print("\nMissing values:")
    display(df.isna().sum())


def file_checksum(path):
    """SHA-256 checksum for reproducibility."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


print("Generic loaders defined.")



# 3. Tabular + Medical Data: UCI Cleveland Heart Disease

We use the commonly distributed Cleveland dataset with 14 clinically relevant columns:

`age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal, target`

The target is converted to a binary variable:

- `0` → no diagnosed heart disease
- `1–4` → presence of heart disease


In [ ]:

# Download Cleveland heart disease data.
# This URL is a commonly used public mirror of the UCI Cleveland data.
heart_url = "https://raw.githubusercontent.com/selva86/datasets/master/Heart.csv"

try:
    heart = pd.read_csv(heart_url)
    print("Downloaded dataset.")
    print("Shape:", heart.shape)
except Exception as e:
    print("Download failed:", e)
    print("You can upload a Cleveland heart-disease CSV to Colab and replace heart_url with its path.")
    heart = pd.DataFrame()

if not heart.empty:
    display(heart.head())
    print("\nColumns:", list(heart.columns))


In [ ]:

# Standardize column names and inspect the downloaded dataset
if not heart.empty:
    heart.columns = [str(c).strip().lower() for c in heart.columns]

    # Handle common naming conventions
    if "aHD".lower() in heart.columns:
        heart = heart.rename(columns={"ahd": "target"})
    elif "target" not in heart.columns:
        # Some mirrors use 'num' or another final target column
        possible_targets = [c for c in heart.columns if c in ["num", "diagnosis", "condition", "hd"]]
        if possible_targets:
            heart = heart.rename(columns={possible_targets[0]: "target"})

    print("Shape:", heart.shape)
    display(heart.head())
    print("\nMissing values:")
    display(heart.isna().sum())
    print("\nData types:")
    display(heart.dtypes)


In [ ]:

# If the mirror has a different schema, create a canonical 14-feature schema.
# This cell also handles the classic UCI processed Cleveland CSV if it is uploaded manually.

canonical_cols = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]

if not heart.empty:
    # Rename common variants
    rename_map = {
        "restbp": "trestbps",
        "restecg": "restecg",
        "maxhr": "thalach",
        "num": "target"
    }
    heart = heart.rename(columns=rename_map)

    # Convert target if necessary
    if "target" in heart.columns:
        heart["target"] = pd.to_numeric(heart["target"], errors="coerce")
        heart["target_binary"] = (heart["target"] > 0).astype("Int64")

    print("Canonicalized dataset preview:")
    display(heart.head())
    print("\nTarget distribution:")
    if "target_binary" in heart:
        print(heart["target_binary"].value_counts(dropna=False))


In [ ]:

# Basic EDA: numeric distributions and correlation matrix
if not heart.empty:
    numeric_cols = heart.select_dtypes(include=np.number).columns.tolist()

    heart[numeric_cols].hist(figsize=(14, 10), bins=15)
    plt.suptitle("Heart Disease Dataset - Numeric Feature Distributions")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 7))
    corr = heart[numeric_cols].corr()
    plt.imshow(corr, aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=90)
    plt.yticks(range(len(numeric_cols)), numeric_cols)
    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.show()


## 4. Baseline Heart Disease Classifier

In [ ]:

# Train a baseline Logistic Regression classifier
if not heart.empty and "target_binary" in heart.columns:
    features = [c for c in canonical_cols[:-1] if c in heart.columns]
    X = heart[features].copy()
    y = heart["target_binary"].copy()

    # Numeric conversion and missing-value handling
    for c in features:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    valid = y.notna()
    X = X.loc[valid]
    y = y.loc[valid].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title("Heart Disease - Confusion Matrix")
    plt.show()

    RocCurveDisplay.from_predictions(y_test, y_prob)
    plt.title("Heart Disease - ROC Curve")
    plt.show()



# 5. Signal Data: MIT-BIH Arrhythmia Database

Record `100` is downloaded through the WFDB/PhysioNet interface.

The notebook:
1. Reads the two-channel ECG signal.
2. Reads expert beat annotations.
3. Displays a short ECG segment.
4. Aligns annotations with the waveform.
5. Calculates basic RR intervals and heart-rate statistics.


In [ ]:

# Download one MIT-BIH record through WFDB
record_name = "100"

try:
    record = wfdb.rdrecord(record_name, pn_dir="mitdb")
    annotation = wfdb.rdann(record_name, "atr", pn_dir="mitdb")

    print("Signal shape:", record.p_signal.shape)
    print("Sampling frequency:", record.fs, "Hz")
    print("Number of annotations:", len(annotation.sample))
    print("Signal names:", record.sig_name)
except Exception as e:
    record = None
    annotation = None
    print("Could not download MIT-BIH record:", e)
    print("Check internet access or run this cell again in Colab.")


In [ ]:

if record is not None:
    # Plot first 10 seconds of channel 0
    seconds = 10
    n_samples = min(int(seconds * record.fs), len(record.p_signal))

    t = np.arange(n_samples) / record.fs

    plt.figure(figsize=(15, 4))
    plt.plot(t, record.p_signal[:n_samples, 0], label="ECG Channel 0")

    ann_mask = annotation.sample < n_samples
    ann_samples = annotation.sample[ann_mask]
    ann_symbols = np.array(annotation.symbol)[ann_mask]

    plt.scatter(
        ann_samples / record.fs,
        record.p_signal[ann_samples, 0],
        marker="o",
        s=35,
        label="Annotated beats"
    )

    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.title("MIT-BIH Record 100 - ECG with Beat Annotations")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    print("First annotation symbols:", list(annotation.symbol[:20]))


In [ ]:

# Basic RR interval and heart-rate statistics
if annotation is not None and record is not None:
    # Use beat annotations commonly representing QRS complexes
    beat_symbols = {"N", "L", "R", "A", "a", "J", "S", "V", "F", "E", "/", "f", "Q"}

    beat_samples = [
        s for s, sym in zip(annotation.sample, annotation.symbol)
        if sym in beat_symbols
    ]

    beat_samples = np.array(beat_samples)

    if len(beat_samples) > 2:
        rr = np.diff(beat_samples) / record.fs
        # Remove extreme intervals for a simple robust demonstration
        rr_clean = rr[(rr > 0.3) & (rr < 2.0)]

        heart_rates = 60 / rr_clean

        print("Number of usable RR intervals:", len(rr_clean))
        print("Mean RR interval:", round(rr_clean.mean(), 3), "seconds")
        print("Median RR interval:", round(np.median(rr_clean), 3), "seconds")
        print("Mean heart rate:", round(heart_rates.mean(), 2), "BPM")
        print("Heart-rate standard deviation:", round(heart_rates.std(), 2), "BPM")

        plt.figure(figsize=(10, 4))
        plt.plot(rr_clean[:100], marker=".")
        plt.xlabel("Beat interval index")
        plt.ylabel("RR interval (seconds)")
        plt.title("First 100 Clean RR Intervals")
        plt.grid(True, alpha=0.3)
        plt.show()
    else:
        print("Not enough annotations for RR analysis.")



# 6. Textual Data: Synthetic Clinical Note

For a safe classroom demonstration, we use a synthetic note rather than real patient information.

The pipeline demonstrates:
- text loading
- cleaning
- tokenization
- word-frequency calculation
- extraction of simple clinically relevant keywords


In [ ]:

clinical_note = """
Patient reports intermittent chest pain during physical activity.
The patient describes mild shortness of breath and occasional fatigue.
No recent fever is reported. Resting heart rate appears elevated.
A follow-up cardiovascular evaluation and ECG assessment are recommended.
""".strip()

# Save and reload to demonstrate a real text-file ingestion step
text_path = "/content/sample_clinical_note.txt"
with open(text_path, "w", encoding="utf-8") as f:
    f.write(clinical_note)

loaded_note = load_text(text_path)

print("Loaded clinical note:\n")
print(loaded_note)


In [ ]:

# Basic text preprocessing and frequency analysis
clean_text = loaded_note.lower()
clean_text = re.sub(r"[^a-z0-9\s-]", " ", clean_text)
tokens = clean_text.split()

stopwords = {
    "the", "a", "an", "and", "or", "of", "to", "is", "are", "was",
    "with", "during", "no", "for", "appears", "be", "has"
}

filtered_tokens = [tok for tok in tokens if tok not in stopwords]
freq = Counter(filtered_tokens)

print("Total tokens:", len(tokens))
print("Tokens after simple stopword removal:", len(filtered_tokens))
print("\nMost common terms:")
for word, count in freq.most_common(15):
    print(f"{word:20s} {count}")

clinical_terms = [
    "chest pain", "shortness of breath", "fatigue",
    "heart rate", "ecg", "cardiovascular"
]

print("\nDetected clinical terms:")
for term in clinical_terms:
    if term in clean_text:
        print("✓", term)



# 7. Image Data Demonstration

To keep the notebook self-contained, a synthetic grayscale medical-style image is generated and then read using Pillow.

For real medical imaging:
- **DICOM** → `pydicom`
- **NIfTI** → `nibabel`
- **PNG/JPEG/TIFF** → `Pillow` / OpenCV

Real medical images may also require metadata handling, intensity normalization, windowing, resizing, and privacy/de-identification workflows.


In [ ]:

# Create a synthetic grayscale image representing a simple medical-style scan
img_array = np.zeros((256, 256), dtype=np.uint8)

yy, xx = np.mgrid[:256, :256]
center_x, center_y = 128, 128
ellipse = ((xx - center_x) / 85) ** 2 + ((yy - center_y) / 105) ** 2

img_array[ellipse < 1] = 90

# Add two simple lung-like regions
left_lung = ((xx - 95) / 38) ** 2 + ((yy - 130) / 70) ** 2
right_lung = ((xx - 161) / 38) ** 2 + ((yy - 130) / 70) ** 2

img_array[left_lung < 1] = 170
img_array[right_lung < 1] = 170

# Add a central structure
heart = ((xx - 128) / 28) ** 2 + ((yy - 155) / 35) ** 2
img_array[heart < 1] = 230

synthetic_path = "/content/synthetic_medical_image.png"
Image.fromarray(img_array).save(synthetic_path)

medical_image = load_image(synthetic_path)

print("Image format:", medical_image.format)
print("Image size:", medical_image.size)
print("Image mode:", medical_image.mode)

plt.figure(figsize=(6, 6))
plt.imshow(medical_image, cmap="gray")
plt.axis("off")
plt.title("Synthetic Medical Image")
plt.show()

# Resize as a simple preprocessing example
resized = medical_image.resize((128, 128))
print("Resized image size:", resized.size)



# 8. Medical Data Validation / Inspection Summary

A multimodal ingestion system should record:
- source and file format
- dimensions / shape
- data types
- missingness
- sampling frequency for signals
- image dimensions/channels
- patient/subject identifiers when available
- timestamps
- preprocessing operations
- checksums and software versions for reproducibility


In [ ]:

# Compact validation summary for the objects loaded in this notebook

summary = {
    "tabular": {
        "loaded": not heart.empty,
        "shape": list(heart.shape) if not heart.empty else None,
        "missing_values": int(heart.isna().sum().sum()) if not heart.empty else None
    },
    "text": {
        "loaded": bool(loaded_note),
        "characters": len(loaded_note),
        "tokens": len(tokens)
    },
    "image": {
        "loaded": medical_image is not None,
        "size": medical_image.size,
        "mode": medical_image.mode
    },
    "signal": {
        "loaded": record is not None,
        "shape": list(record.p_signal.shape) if record is not None else None,
        "sampling_frequency_hz": float(record.fs) if record is not None else None,
        "annotations": len(annotation.sample) if annotation is not None else None
    }
}

print(json.dumps(summary, indent=2))



# 9. Multimodal Fusion Interface

A future cardiovascular risk model could combine modalities as follows:

**Patient ID / Timestamp**
→ Tabular features (age, BP, cholesterol, etc.)  
→ ECG features (heart rate, RR intervals, rhythm features)  
→ Clinical-note features (symptoms and context)  
→ Image features (if relevant imaging is available)  
→ **Feature Fusion Layer**  
→ Classification / Risk Prediction

### Important integration considerations
1. Use a consistent patient/subject identifier.
2. Align measurements using timestamps where appropriate.
3. Keep train/validation/test splits at the patient level to prevent leakage.
4. Handle missing modalities explicitly.
5. De-identify sensitive medical information.
6. Store preprocessing and dataset versions for reproducibility.


In [ ]:

# Example conceptual fusion record for ONE subject
# This is only an interface demonstration, not a trained multimodal model.

fusion_example = {
    "subject_id": "SYNTHETIC_001",
    "tabular_features": {
        "age": 55,
        "sex": 1,
        "cholesterol": 230
    },
    "ecg_features": {
        "mean_rr_seconds": 0.82,
        "mean_heart_rate_bpm": 73.2
    },
    "text_features": {
        "symptoms": ["chest pain", "fatigue"],
        "ecg_mentioned": True
    },
    "image_features": {
        "image_available": True,
        "width": 256,
        "height": 256
    }
}

display(pd.DataFrame([fusion_example]))
print("Conceptual multimodal record created successfully.")



# 10. Final Result / Conclusion

This notebook demonstrates a modular ingestion pipeline for:

| Modality | Example format/source | Python tool |
|---|---|---|
| Tabular | CSV / UCI Cleveland | pandas |
| Text | TXT clinical note | Python / regex |
| Image | PNG | Pillow |
| Signal | MIT-BIH `.dat/.hea/.atr` | WFDB |
| Medical composite | Tabular + ECG + text + image | pandas + WFDB + Pillow |

### Completed tasks
- Installed the required Python libraries.
- Loaded and inspected tabular medical data.
- Handled missing values and converted the target to binary where applicable.
- Performed basic EDA.
- Trained a Logistic Regression baseline and evaluated Accuracy, ROC-AUC, classification report, confusion matrix, and ROC curve.
- Loaded an MIT-BIH ECG record and annotations.
- Visualized ECG waveform and beat annotations.
- Computed basic RR interval and heart-rate statistics.
- Loaded and processed a synthetic clinical text file.
- Performed tokenization, stopword removal, word-frequency analysis, and keyword extraction.
- Created, loaded, displayed, and resized a synthetic medical image.
- Built a validation summary.
- Documented a multimodal fusion interface for future research.

**Note:** This is an educational data-ingestion and baseline-analysis notebook, not a clinical diagnostic system.
